# Skills Workflow

One notebook for the full skill lifecycle: **Ingest** -> **Update from source** -> **Assign destinations** -> **Distribute**.

Use the phase selector below to jump to what you need this run — most weeks you'll only touch one or two phases. Each phase reads its own state fresh from disk (manifests, CSV), so phases don't depend on a prior phase's in-memory state and can be run in any combination.

**"Run Selected" contract:** Phase 4 (Distribute) runs to completion in one click — it has no human decision points. Phases 1–3 inherently require a human to review a diff/report and make a per-item choice, so "Run Selected" for those runs the first read-only step (scan/check) and surfaces the manual widgets in that phase's own section below. It does not claim to fully automate a decision only you can make.

Supersedes `ingest-project.ipynb`, `update-sourced-skills.ipynb`, and `sync_orchestrator.ipynb` (all three retired in favor of this notebook).

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "manifests" / "origins.json").is_file():
    for p in [REPO_ROOT, *REPO_ROOT.parents]:
        if (p / "manifests" / "origins.json").is_file():
            REPO_ROOT = p
            break
    else:
        raise FileNotFoundError("manifests/origins.json not found")

sys.path.insert(0, str(REPO_ROOT))

import ipywidgets as widgets
from IPython.display import display

from scripts import ingest_engine
from scripts import update_engine
from scripts.sync_engine import (
    load_env, load_origins, load_destinations,
    build_prompt_cache, distribute_all, format_report,
)
from scripts.generate_destinations_csv import generate_csv
from scripts.update_destinations_from_csv import update_destinations_from_csv

PHASE_RUNNERS = {}

print(f"Repo root: {REPO_ROOT}")

In [ ]:
phase_checkboxes = {
    1: widgets.Checkbox(value=False, description="Phase 1 — Ingest"),
    2: widgets.Checkbox(value=False, description="Phase 2 — Update from source"),
    3: widgets.Checkbox(value=False, description="Phase 3 — Assign destinations"),
    4: widgets.Checkbox(value=False, description="Phase 4 — Distribute"),
}
run_selected_button = widgets.Button(description="Run Selected", button_style="primary")
run_selected_output = widgets.Output()

def on_run_selected_clicked(b):
    run_selected_output.clear_output()
    with run_selected_output:
        selected = [n for n in sorted(phase_checkboxes) if phase_checkboxes[n].value]
        if not selected:
            print("No phases selected.")
            return
        for n in selected:
            runner = PHASE_RUNNERS.get(n)
            if runner is None:
                print(f"Phase {n}: not ready yet — run the notebook cells below top to bottom at least once.")
                continue
            print(f"--- Running Phase {n} ---")
            runner()

run_selected_button.on_click(on_run_selected_clicked)

display(widgets.VBox([*phase_checkboxes.values(), run_selected_button, run_selected_output]))

## Phase 1 — Ingest

Scan a project's skill directory vs. central, classify new skills, reconcile changed ones, apply, then verify the manifest ledger.

In [ ]:
source_path_input = widgets.Text(
    value="", placeholder="C:/path/to/project/.agents/skills",
    description="Source path:", style={"description_width": "150px"},
    layout=widgets.Layout(width="600px"),
)
destination_id_input = widgets.Text(
    value="", placeholder="campus-profile",
    description="Destination ID:", style={"description_width": "150px"},
    layout=widgets.Layout(width="300px"),
)
scan_output = widgets.Output()
scan_button = widgets.Button(description="Scan", button_style="info")

_phase1_state = {}

def on_scan_clicked(b):
    scan_output.clear_output()
    with scan_output:
        source = Path(source_path_input.value).expanduser()
        dest_id = destination_id_input.value.strip()
        if not source.is_dir():
            print(f"Source not found: {source}")
            return
        if not dest_id:
            print("Destination ID required")
            return
        result = ingest_engine.scan_project(REPO_ROOT, source, dest_id)
        _phase1_state["scan"] = result
        print(ingest_engine.format_scan_report(result))

scan_button.on_click(on_scan_clicked)

def run_phase_1():
    if not source_path_input.value.strip() or not destination_id_input.value.strip():
        print("Fill in Source path + Destination ID in the Phase 1 section below, then click Scan.")
        return
    on_scan_clicked(None)
    print("Scan complete — continue with the Classify / Reconcile / Apply / Verify widgets below.")

PHASE_RUNNERS[1] = run_phase_1

display(widgets.VBox([source_path_input, destination_id_input, scan_button, scan_output]))

In [ ]:
classify_output = widgets.Output()
classifications = {}

def build_classify_widgets(b=None):
    classify_output.clear_output()
    with classify_output:
        scan = _phase1_state.get("scan")
        if scan is None or not scan.diff["new"]:
            print("No new skills to classify. Run Scan first.")
            return

        classifications.clear()
        for skill_name in scan.diff["new"]:
            print(f"\n{skill_name}\n")
            choice_radio = widgets.RadioButtons(
                options=[
                    ("[T] Tracked upstream (auto-updated)", "tracked"),
                    ("[F] Fork (modified from upstream)", "fork"),
                    ("[O] Own (no upstream source)", "own"),
                    ("[S] Skip (don't import)", "skip"),
                ],
                layout=widgets.Layout(width="400px"),
            )
            repo_field = widgets.Text(placeholder="https://github.com/...", description="Repo:", style={"description_width": "80px"})
            branch_field = widgets.Text(value="main", description="Branch:", style={"description_width": "80px"})
            subpath_field = widgets.Text(placeholder=f".agents/skills/{skill_name}", description="Subpath:", style={"description_width": "80px"})
            author_field = widgets.Text(placeholder="Author", description="Author:", style={"description_width": "80px"})
            origin_field = widgets.Text(placeholder="e.g., microsoft/fabric-apps", description="Origin:", style={"description_width": "80px"})
            own_field = widgets.Text(placeholder="Owner/author", description="Owner:", style={"description_width": "80px"})

            classifications[skill_name] = {
                "choice": choice_radio, "repo": repo_field, "branch": branch_field,
                "subpath": subpath_field, "author": author_field,
                "origin": origin_field, "owner": own_field,
            }

            display(choice_radio)
            display(widgets.VBox([repo_field, branch_field, subpath_field, author_field, origin_field, own_field]))

build_classify_button = widgets.Button(description="Build Classification Widgets", button_style="warning")
build_classify_button.on_click(build_classify_widgets)

display(build_classify_button)
display(classify_output)

In [ ]:
reconcile_output = widgets.Output()
reconciliations = {}

def build_reconcile_widgets(b=None):
    reconcile_output.clear_output()
    with reconcile_output:
        scan = _phase1_state.get("scan")
        if scan is None or not scan.changed:
            print("No changed skills to reconcile.")
            return

        reconciliations.clear()
        for skill_name in scan.changed:
            repo_path = scan.repo_skills[skill_name]
            project_path = scan.project_skills[skill_name]
            print(f"\n{skill_name}")
            print(f"\n{ingest_engine.show_diff(repo_path, project_path, skill_name)}\n")
            choice_radio = widgets.RadioButtons(
                options=[
                    ("[C] Keep central version (project drifted)", "central"),
                    ("[P] Take project version (intentional edits)", "project"),
                    ("[S] Skip (leave as-is)", "skip"),
                ],
                layout=widgets.Layout(width="400px"),
            )
            reconciliations[skill_name] = choice_radio
            display(choice_radio)

build_reconcile_button = widgets.Button(description="Build Reconciliation Widgets", button_style="warning")
build_reconcile_button.on_click(build_reconcile_widgets)

display(build_reconcile_button)
display(reconcile_output)

In [ ]:
target_dir_input = widgets.Text(
    value="", placeholder="C:/path/to/project/.agents/skills",
    description="Target dir:", style={"description_width": "150px"},
    layout=widgets.Layout(width="600px"),
)
apply_output = widgets.Output()
apply_button = widgets.Button(description="Apply Changes", button_style="success")

def on_apply_clicked(b):
    apply_output.clear_output()
    with apply_output:
        scan = _phase1_state.get("scan")
        if scan is None:
            print("No scan in progress. Run Scan first.")
            return

        classifications_values = {
            name: {
                "choice": c["choice"].value,
                "repo": c["repo"].value,
                "branch": c["branch"].value,
                "subpath": c["subpath"].value,
                "author": c["author"].value,
                "license": "",
                "reason": (
                    f"Fork of {c['origin'].value}; locally modified, do not auto-update."
                    if c["choice"].value == "fork"
                    else f"Own skill ({c['owner'].value or 'Project'}); no upstream source."
                    if c["choice"].value == "own"
                    else ""
                ),
            }
            for name, c in classifications.items()
        }
        reconciliations_values = {name: r.value for name, r in reconciliations.items()}

        summary = ingest_engine.apply_ingest(
            REPO_ROOT, scan, classifications_values, reconciliations_values,
            target_dir_input.value.strip(),
        )
        print("Changes applied:")
        print(f"  Imported: {summary['imported_new']} new + {summary['imported_changed']} changed")
        print(f"  Assigned: {len(summary['skills_assigned'])} skills to {summary['destination_id']}")
        print(f"  Destination: {summary['destination_id']} (DISABLED — enable when ready)")
        print("  Manifests updated: origins.json, destinations.json")

apply_button.on_click(on_apply_clicked)

display(widgets.VBox([target_dir_input, apply_button, apply_output]))

In [ ]:
verify_output = widgets.Output()
verify_button = widgets.Button(description="Verify Skills Ledger", button_style="info")

def on_verify_clicked(b):
    verify_output.clear_output()
    with verify_output:
        result = ingest_engine.verify_ledger(REPO_ROOT)
        print(ingest_engine.format_verify_report(result))

verify_button.on_click(on_verify_clicked)

display(widgets.VBox([verify_button, verify_output]))

## Phase 2 — Update from Source

Checks each tracked skill in `manifests/origins.json` against its upstream repo, shows a change-list (with diffs), and lets you apply or skip each one.

In [ ]:
show_diffs_checkbox = widgets.Checkbox(value=True, description="Show diffs")
check_output = widgets.Output()
check_button = widgets.Button(description="Check for Updates", button_style="info")

_phase2_state = {}

def on_check_clicked(b):
    check_output.clear_output()
    with check_output:
        results = update_engine.check_all(REPO_ROOT)
        _phase2_state["results"] = results
        print(update_engine.format_update_report(results, show_diffs=show_diffs_checkbox.value))

check_button.on_click(on_check_clicked)

def run_phase_2():
    on_check_clicked(None)
    print("Check complete — continue with the Decide/Apply widgets below.")

PHASE_RUNNERS[2] = run_phase_2

display(widgets.VBox([show_diffs_checkbox, check_button, check_output]))

In [ ]:
decision_output = widgets.Output()
decision_widgets = {}

def build_decision_widgets(b=None):
    decision_output.clear_output()
    with decision_output:
        results = _phase2_state.get("results")
        if not results:
            print("No results. Run Check for Updates first.")
            return

        decision_widgets.clear()
        updatable = {name: rep for name, rep in results.items() if rep.has_update}
        if not updatable:
            print("Nothing to decide — all sourced skills are up to date.")
            return

        for name in updatable:
            print(f"\n{name}")
            radio = widgets.RadioButtons(
                options=[("Skip", "skip"), ("Apply", "apply")],
                value="skip",
                layout=widgets.Layout(width="200px"),
            )
            decision_widgets[name] = radio
            display(radio)

build_decision_button = widgets.Button(description="Build Decision Widgets", button_style="warning")
build_decision_button.on_click(build_decision_widgets)

display(build_decision_button)
display(decision_output)

In [ ]:
apply_updates_output = widgets.Output()
apply_updates_button = widgets.Button(description="Apply Decisions", button_style="success")

def on_apply_updates_clicked(b):
    apply_updates_output.clear_output()
    with apply_updates_output:
        results = _phase2_state.get("results")
        if not results:
            print("No results. Run Check for Updates first.")
            return
        decisions = {name: radio.value for name, radio in decision_widgets.items()}
        outcome = update_engine.apply_decisions(REPO_ROOT, results, decisions)
        print(f"Applied: {outcome['applied'] or 'none'}")
        print(f"Skipped: {outcome['skipped'] or 'none'}")
        if outcome["applied"]:
            print("Stamped origins.json with new last_synced_sha values.")

        # Also stamp last_synced_sha for skills confirmed up to date this run
        # (not just applied ones) — otherwise the ref-only pre-check in
        # check_all() never has anything to compare against for skills that
        # simply had no changes, and re-clones them every time regardless.
        stamped = update_engine.stamp_synced_shas(REPO_ROOT, results)
        if stamped:
            print(f"Also confirmed in sync (no changes): {stamped}")

apply_updates_button.on_click(on_apply_updates_clicked)

display(widgets.VBox([apply_updates_button, apply_updates_output]))

## Phase 3 — Assign Destinations

This phase is a deliberate handoff to a spreadsheet, not an in-notebook grid editor — Excel already handles a skill × destination assignment matrix well, and building a data-grid widget here would be over-engineering.

1. Click **Export CSV** — writes `destinations-matrix.csv`.
2. Edit it in Excel: put `x` in a cell to assign that skill to that destination, add a new column for a new destination.
3. Click **Import CSV** — rewrites `manifests/destinations.json` from your edits.

In [ ]:
csv_output = widgets.Output()
export_csv_button = widgets.Button(description="Export CSV", button_style="info")
import_csv_button = widgets.Button(description="Import CSV", button_style="success")

def on_export_csv_clicked(b):
    csv_output.clear_output()
    with csv_output:
        path = generate_csv(REPO_ROOT)
        print(f"Wrote {path}")
        print("Edit it in Excel, then click Import CSV.")

def on_import_csv_clicked(b):
    csv_output.clear_output()
    with csv_output:
        manifest = update_destinations_from_csv(REPO_ROOT)
        for dest in manifest["destinations"]:
            print(f"  {dest['id']}: {len(dest['skills_assigned'])} skills assigned, enabled={dest.get('enabled', True)}")

export_csv_button.on_click(on_export_csv_clicked)
import_csv_button.on_click(on_import_csv_clicked)

def run_phase_3():
    print("Phase 3 is a spreadsheet handoff: click Export CSV, edit destinations-matrix.csv in Excel, then click Import CSV below.")

PHASE_RUNNERS[3] = run_phase_3

display(widgets.VBox([export_csv_button, import_csv_button, csv_output]))

## Phase 4 — Distribute

Fully automatable, no human decision points: builds the prompt cache and pushes it to every enabled destination.

In [ ]:
distribute_output = widgets.Output()
distribute_button = widgets.Button(description="Distribute", button_style="success")

def on_distribute_clicked(b):
    distribute_output.clear_output()
    with distribute_output:
        env = load_env(REPO_ROOT)
        origins = load_origins(REPO_ROOT)
        destinations = load_destinations(REPO_ROOT)
        cache_dir = REPO_ROOT / ".cache"

        cached = build_prompt_cache(REPO_ROOT, origins)
        print(f"Cached {len(cached)} skill prompt(s)")

        results = distribute_all(destinations, cache_dir, REPO_ROOT)
        print(format_report(None, results))

distribute_button.on_click(on_distribute_clicked)

def run_phase_4():
    on_distribute_clicked(None)

PHASE_RUNNERS[4] = run_phase_4

display(widgets.VBox([distribute_button, distribute_output]))